In [ ]:
import pandas as pd

#get cg74 related cgs in a list
file_path = "../inflammation_HANNUM_with_proper_selection/cd_74_associated_cpgs.txt"
with open(file_path, "r") as file:
    cd_74_associated_cpgs_list = [line.strip() for line in file]  # Removes any trailing newlines
cd_74_associated_cpgs_set=set(cd_74_associated_cpgs_list)
#read meta
train_meta = pd.read_csv('computage_train_meta.tsv', sep='\t', index_col=0).T
train_meta.columns = train_meta.columns.str.lower()
#delete duplicate cols
train_meta = train_meta.loc[:, ~train_meta.columns.duplicated(keep='first')]

In [ ]:
#Dont run unnecessary, since it takes a lot of time to read
train_betas_UNMODIFIABLE = pd.read_pickle('train_betas.pkl') #takes 15m to read
#drop duplicate cols
train_betas_UNMODIFIABLE = train_betas_UNMODIFIABLE.loc[:, ~train_betas_UNMODIFIABLE.columns.duplicated(keep='first')]
train_betas = train_betas_UNMODIFIABLE.copy()

In [4]:
#keep only rows which are in the infl_cg_set
train_betas = train_betas[train_betas.index.isin(cd_74_associated_cpgs_set)] #all the cg sites are present
# Drop columns with more than 250 missing values
train_betas = train_betas.loc[:, train_betas.isnull().sum() <= 250]
# Drop rows with more than 2500 missing values
train_betas = train_betas.loc[train_betas.isnull().sum(axis=1) <= 2500]
#fill nan values with means
train_betas.fillna(train_betas.mean(), inplace=True)
#the raw and meta has to be in the same column order
train_meta = train_meta[list(train_betas.columns)]

In [ ]:
import torch
from torch.utils.data import Dataset

#create a dataset class
class MethylationDataset(Dataset):
    def __init__(self, meta_table, raw_table, transform=None):
        self.meta_table = meta_table
        self.raw_table = raw_table
        
    def __len__(self):
        return self.raw_table.shape[1]

    def __getitem__(self, idx):
        # Ensure the index is within the bounds of the meta_table
        if idx >= self.__len__():
            print('Index out of bounds:', idx)
            return -1

        # Extract the RNA data and age
        try:
            rna_data = self.raw_table.iloc[:, idx].tolist()
            age=float(self.meta_table.iloc[5,idx])
        except IndexError as e:
            print(f"IndexError: {e}, idx: {idx}")
            return -1
        return torch.tensor(rna_data), age
    
    def get_features_and_targets(self):
        features = []
        targets = []
        for idx in range(len(self)):
            rna_data, age = self[idx]
            features.append(rna_data.numpy())
            targets.append(age.numpy())
        return np.array(features), np.array(targets)
        

In [ ]:
from sklearn.model_selection import train_test_split

#create the dataset object, and random train test split 80-20
transcriptomic_dataset = MethylationDataset(train_meta,train_betas)
train_data, test_data = train_test_split(transcriptomic_dataset, test_size=0.2, random_state=42)

print("Train data size:", len(train_data))
print("Test data size:", len(test_data))

Train data size: 4831
Test data size: 1208


In [ ]:
# Initialize lists for features and targets
X_train = []
y_train = []

for features, target in train_data:
    X_train.append(features)
    y_train.append(target)

X_test = []
y_test = []

for features, target in test_data:
    X_test.append(features)
    y_test.append(target)

In [ ]:
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

pipeline_xgb = Pipeline([
    ('regressor', XGBRegressor(objective='reg:squarederror', random_state=0))
])
#define the parameter grid
param_grid_xgb = {
    'regressor__n_estimators': [300, 500, 600],
    'regressor__max_depth': [3,5],
    'regressor__learning_rate': [0.1, 0.05],
    'regressor__reg_alpha': [0, 0.2],     # L1 regularization
    'regressor__reg_lambda': [1, 5]       # L2 regularization
}

grid_xgb = GridSearchCV(pipeline_xgb, param_grid_xgb, cv=3, 
                        scoring='neg_mean_absolute_error', verbose=1, n_jobs=4)
grid_xgb.fit(X_train, y_train)
print("Best XGBoost params:", grid_xgb.best_params_)
#grid search best model
best_model = grid_xgb.best_estimator_



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
import joblib

#grid search best model
best_model = grid_xgb.best_estimator_

#Predict on the test set
test_predictions = best_model.predict(X_test)

#evaluating
mse = mean_squared_error(y_test, test_predictions)
mae = mean_absolute_error(y_test, test_predictions)
pearson_r, _ = pearsonr(y_test, test_predictions)

print("TEST Mean Squared Error:", mse)
print("Mean Absolute Error:", mae)
print("Pearson correlation coefficient:", pearson_r)

#save the model
joblib.dump(best_model, 'xgboost_model_2.pkl')

# Plot: Predicted vs True
plt.figure(figsize=(6,6))
plt.scatter(y_test, test_predictions, alpha=0.6)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')  # y=x referencia
plt.xlabel('True Age')
plt.ylabel('Predicted Age')
plt.title('XGBoost Predictions vs True Age')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create scatter plot
plt.scatter(y_test, test_predictions, color='blue', label='Data points')

# Add the f(x) = x line THE REAL AGE THAT THE BLUE DOTS SHOULD CONVERGE TO
x_line = np.linspace(0, 100, 400)
y_line = x_line  # Since f(x) = x
plt.plot(x_line, y_line, color='red', label='f(x) = x')

# Add labels and title
plt.xlabel('X Values')
plt.ylabel('Y Values')
plt.title('Scatter Plot of X vs Y')
plt.xlim(0, 100)  # Set the limits for x-axis
plt.ylim(0, 100)  # Set the limits for y-axis
plt.legend()

# Show plot
plt.grid(True)
plt.show()